In [92]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import os

In [93]:
# Change crop name each time
CROP_NAME = "Brinjal"

train_dir = f"/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/split_dataset/train/{CROP_NAME}"
val_dir   = f"/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/split_dataset/val/{CROP_NAME}"
test_dir  = f"/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/split_dataset/test/{CROP_NAME}"

In [94]:
IMG_SIZE = 224
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 89 images belonging to 5 classes.
Found 45 images belonging to 5 classes.


In [95]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))

# Reduce effect of weights > 3
class_weights = {k: v * 0.7 if v > 3 else v for k, v in class_weights.items()}

print(class_weights)

{0: np.float64(1.78), 1: np.float64(1.0470588235294118), 2: np.float64(0.445), 3: np.float64(1.78), 4: np.float64(1.4833333333333334)}


In [96]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base layers
for layer in base_model.layers[:-15]:
    layer.trainable = False

for layer in base_model.layers[-15:]:
    layer.trainable = True

In [97]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [98]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [100]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

In [101]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 60s 10s/step - accuracy: 0.1798 - loss: 2.0119 - val_accuracy: 0.5333 - val_loss: 1.4834
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.3483 - loss: 1.5283 - val_accuracy: 0.6222 - val_loss: 1.1305
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.4382 - loss: 1.2375 - val_accuracy: 0.7556 - val_loss: 0.9205
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.6292 - loss: 1.0034 - val_accuracy: 0.7778 - val_loss: 0.7997
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.6742 - loss: 0.9557 - val_accuracy: 0.7778 - val_loss: 0.7001
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.7640 - loss: 0.6425 - val_accuracy: 0.7778 - val_loss: 0.6299
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.8315 - loss: 0.5581 - val_accuracy: 0.7778 - val_loss: 0.5894
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.8202 - loss: 0.5259 - val_accuracy: 0.7778 - val_loss: 0.5663
Epoch 9/20
6/6

In [103]:
model.save(f"/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/Train_model/{CROP_NAME}_model.keras")

In [102]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

Found 52 images belonging to 5 classes.
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.9423 - loss: 0.2259
Test Accuracy: 0.942307710647583


In [104]:
print(train_data.class_indices)

{'Bacrerial_Wilt': 0, 'Cercospora_Leaf_spot': 1, 'Healthy': 2, 'Mosaic': 3, 'Phomopsis_Leaf_Blight': 4}


In [105]:
import os

path = "/content/drive/MyDrive/Ganpat_hackathon_2026/ML1/Train_model"
print(os.listdir(path))

['Castor_model.keras', 'Brinjal_model.keras', 'Papaya_model.keras', 'Guava_model.keras', 'Cumin_model.keras']
